# CardioVis FixMatch Train (dat_kim_goc) — ablation study

**How to run**
1. Runtime → Change runtime type → **GPU**
2. Run the mount/setup cell once
3. Edit only the **CONFIG** cell (ablation toggles + `EXP_NAME` / `MODEL` / `FOLDS`)
4. Runtime → **Run all**

**Shared:** background is always excluded from train loss and metrics.

**Ablation toggles** (any combination; rename `EXP_NAME` each run):
| Run | Dense U | Weights | High-res | Suggested EXP_NAME |
|-----|---------|---------|----------|--------------------|
| baseline | off | off | off | `abl_baseline_dlv3` |
| +m1 | on | off | off | `abl_m1_dlv3` |
| +m2 | off | on | off | `abl_m2_dlv3` |
| +m3 | off | off | on | `abl_m3_dlv3` |
| full | on | on | on | `abl_full_dlv3` |

Allowed `MODEL`: `deeplabv3plus_resnet101` | `unet_resnet34` | `segformer_b2`

`FOLDS`: list of ints, e.g. `[1, 2, 3, 4, 5]` or just `[1]` for a smoke run.

In [ ]:
# Setup (do not edit)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, subprocess

ROOT = "/content/drive/MyDrive/Cardiovis-related"
CODE = f"{ROOT}/AI-development"
assert os.path.isdir(CODE), f"Missing code at {CODE} — rclone sync AI-development first"
os.chdir(CODE)
sys.path.insert(0, CODE)

pkgs = [
    "monai",
    "segmentation-models-pytorch",
    "albumentations",
    "tqdm",
    "pyyaml",
    "opencv-python-headless",
    "scipy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("cwd:", os.getcwd())

In [ ]:
# === CHỈ SỬA CELL NÀY ===
MODEL = "deeplabv3plus_resnet101"  # unet_resnet34 | segformer_b2
EXP_NAME = "abl_baseline_dlv3"     # đổi theo ablation để không ghi đè checkpoint
FOLDS = [1, 2, 3, 4, 5]            # hoặc [1] để chạy nhanh 1 fold

# Shared (keep True for this study)
EXCLUDE_BACKGROUND = True

# Ablation toggles (any combination)
USE_DENSE_UNLABELED = False   # method 1 → unlabeled_data_dense
USE_CLASS_WEIGHTS = False     # method 2 → weighted CE + rare oversample
USE_HIGH_RES_RETUNE = False   # method 3 → 384 / conf 0.90 / 60 epochs

# Batches (same for all ablations)
L_BATCH = 32
U_BATCH = 32

# Resolved from USE_HIGH_RES_RETUNE
if USE_HIGH_RES_RETUNE:
    IMG_SIZE, CONF_THRESH, NUM_EPOCHS, WARMUP_EPOCHS = 384, 0.90, 60, 15
else:
    IMG_SIZE, CONF_THRESH, NUM_EPOCHS, WARMUP_EPOCHS = 224, 0.95, 30, 15

ROOT = "/content/drive/MyDrive/Cardiovis-related"
CODE = f"{ROOT}/AI-development"
DATA = f"{ROOT}/training_related/dat_kim_goc"
CONFIG_YML = f"{CODE}/Configs/cardio.yml"
UNLABELED = (
    f"{DATA}/unlabeled_data_dense" if USE_DENSE_UNLABELED else f"{DATA}/unlabeled_data"
)

print(
    f"EXP={EXP_NAME} | dense_u={USE_DENSE_UNLABELED} | weights={USE_CLASS_WEIGHTS} | "
    f"high_res={USE_HIGH_RES_RETUNE} | img={IMG_SIZE} conf={CONF_THRESH} "
    f"epochs={NUM_EPOCHS} batch={L_BATCH}/{U_BATCH}"
)
print(f"unlabeled → {UNLABELED}")

In [ ]:
# Train (do not edit) — streams logs/tqdm live into the cell output
import os, sys, subprocess

os.chdir(CODE)
for p in (
    f"{DATA}/labeled_data",
    UNLABELED,
    f"{DATA}/test_data",
    CONFIG_YML,
    f"{CODE}/fixmatch.py",
):
    assert os.path.exists(p), f"Missing: {p}"
assert os.path.isfile(f"{UNLABELED}/unlabeled.txt"), f"Missing unlabeled.txt in {UNLABELED}"

_folds = FOLDS if isinstance(FOLDS, (list, tuple)) else [FOLDS]
_folds = [int(f) for f in _folds]

cmd = [
    sys.executable, "-u", "fixmatch.py",
    "--config_yml", CONFIG_YML,
    "--model", MODEL,
    "--exp", EXP_NAME,
    "--folds", *[str(f) for f in _folds],
    "--num_epochs", str(NUM_EPOCHS),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--l_batchsize", str(L_BATCH),
    "--u_batchsize", str(U_BATCH),
    "--img_size", str(IMG_SIZE),
    "--conf_thresh", str(CONF_THRESH),
    "--data_root", DATA,
    "--unlabeled_folder", UNLABELED,
    "--exclude_background", "1" if EXCLUDE_BACKGROUND else "0",
    "--use_class_weights", "1" if USE_CLASS_WEIGHTS else "0",
]
print("Running:", " ".join(cmd), flush=True)
print(
    f"Folds={_folds} | epochs={NUM_EPOCHS} | model={MODEL} | "
    f"dense_u={USE_DENSE_UNLABELED} | weights={USE_CLASS_WEIGHTS} | high_res={USE_HIGH_RES_RETUNE}",
    flush=True,
)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TQDM_MININTERVAL"] = "1"

proc = subprocess.Popen(
    cmd,
    cwd=CODE,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"fixmatch.py exited with code {rc}")
print("Training finished OK.", flush=True)

In [ ]:
# Results (do not edit)
from pathlib import Path

_folds = FOLDS if isinstance(FOLDS, (list, tuple)) else [FOLDS]
_folds = [int(f) for f in _folds]
base = Path(CODE) / "checkpoints" / "dat_kim_goc" / EXP_NAME

for fold in _folds:
    results = base / f"fold{fold}" / "test_results.txt"
    print("=" * 60)
    print(f"Fold {fold}: {results}")
    if results.is_file():
        print(results.read_text())
    else:
        print("No test_results.txt yet")
    log = results.parent / "log.txt"
    if log.is_file():
        lines = log.read_text().strip().splitlines()
        print("--- last log lines ---")
        print("\n".join(lines[-10:]))